# ZeusDL — Agent Hermes sur Google Colab

**Etapes :**
1. Installer ZeusDL
2. Configurer (email + mot de passe BangBros, bot Telegram)
3. Se connecter automatiquement — pas besoin d'exporter des cookies
4. Demarrer l'agent et attendre des ordres


In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 1 — Installation
# ═══════════════════════════════════════════════════════════
import subprocess, sys

print('Installation de ZeusDL depuis GitHub...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
     'git+https://github.com/ferelking242/zeusdl.git@main#subdirectory=zeusdl'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('OK — ZeusDL installe')
else:
    print('Erreur GitHub, tentative PyPI...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'zeusdl'])
    print('OK — ZeusDL installe depuis PyPI')

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 2 — Configuration
# ═══════════════════════════════════════════════════════════

# ── Agent Hermes ────────────────────────────────────────────
AGENT_ID      = 'colab-1'
MASTER_URL    = 'https://7262cf76-4386-46c0-a4c6-db2e48981e1f-00-1sts3c4c5faag.kirk.replit.dev'

# ── Telegram ─────────────────────────────────────────────────
BOT_TOKEN     = '7909647438:AAFUtLLCo8GOuo8yjgs0C2adGsZeNVBsopw'
CHANNEL       = ''   # ex: '-1001234567890'

# ── BangBros — email + mot de passe (pas besoin de cookies) ──
BB_EMAIL      = 'aivos.dev@gmail.com'
BB_PASSWORD   = 'ferelONDONGO1631'

# ── Telechargements ──────────────────────────────────────────
OUTPUT_DIR    = '/content/downloads'
QUALITY       = '1080p'
WORKERS       = 3
COOKIES_PATH  = '/content/bangbros_cookies.txt'

# ─────────────────────────────────────────────────────────────
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

from zeusdl.config_manager import ZeusConfig
cfg = ZeusConfig()
cfg.set_many({
    'telegram.bot_token':       BOT_TOKEN,
    'telegram.default_channel': CHANNEL,
    'bangbros.username':        BB_EMAIL,
    'bangbros.password':        BB_PASSWORD,
    'bangbros.cookies_path':    COOKIES_PATH,
    'defaults.quality':         QUALITY,
    'defaults.workers':         WORKERS,
    'defaults.output_dir':      OUTPUT_DIR,
    'hermes.master_url':        MASTER_URL,
    'hermes.agent_id':          AGENT_ID,
})
print(f'Config OK — agent: {AGENT_ID} | qualite: {QUALITY} | sortie: {OUTPUT_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 3 — Connexion automatique BangBros
#              (plus besoin d'exporter des cookies !)
# ═══════════════════════════════════════════════════════════
from zeusdl.bangbros_login import BangBrosLogin

login = BangBrosLogin(cookies_path=COOKIES_PATH)
cookies_file = login.login(BB_EMAIL, BB_PASSWORD)
print(f'Connexion OK — cookies sauvegardes dans {cookies_file}')

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 4 — Verification mise a jour
# ═══════════════════════════════════════════════════════════
from zeusdl.hermes.updater import AgentUpdater

up = AgentUpdater()
if up.check():
    print('Mise a jour disponible — installation...')
    up.update()
    print('Agent mis a jour. Relance les cellules depuis le debut.')
else:
    print('Agent a jour.')

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 5 — Demarrer l'agent Hermes
#              (se connecte au Mastermind sur Replit)
# ═══════════════════════════════════════════════════════════
from zeusdl.hermes.agent import HermesAgent

agent = HermesAgent(agent_id=AGENT_ID, master_url=MASTER_URL)
agent.set_var('quality',        QUALITY)
agent.set_var('workers',        str(WORKERS))
agent.set_var('cookies',        COOKIES_PATH)
agent.set_var('output',         OUTPUT_DIR)
agent.set_var('telegram_token', BOT_TOKEN)
agent.set_var('telegram_channel', CHANNEL)

print(f'Agent {AGENT_ID} connecte au Mastermind...')
print()
print('Ordres disponibles depuis @TGBOX_HQbot :')
print(f'  /send {AGENT_ID} download https://site-ma.bangbros.com/scenes?addon=5971')
print(f'  /send {AGENT_ID} push telegram')
print(f'  /agents   <- voir les agents connectes')
print()
agent.start()   # bloquant — laisse tourner

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 6 — Download manuel (optionnel, sans Mastermind)
# ═══════════════════════════════════════════════════════════
from zeusdl.zscript.runner import ZeusScriptRunner

SCRIPT = f"""
set quality   {QUALITY}
set workers   {WORKERS}
set output    {OUTPUT_DIR}
set cookies   {COOKIES_PATH}

download https://site-ma.bangbros.com/scenes?addon=5971

push telegram
    token    {BOT_TOKEN}
    channel  {CHANNEL or 'METS_TON_CHANNEL_ICI'}
    message  Batch Colab {QUALITY} termine
"""

# Decommenter pour lancer :
# ZeusScriptRunner().run_string(SCRIPT)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  CELLULE 7 — Raccourcis Python directs
# ═══════════════════════════════════════════════════════════

# Telecharger seulement
from zeusdl.bulk_download import BulkDownloader
dl = BulkDownloader(output_dir=OUTPUT_DIR, cookies=COOKIES_PATH,
                    workers=WORKERS, quality=QUALITY)
# dl.run('https://site-ma.bangbros.com/scenes?addon=5971')

# Uploader vers Telegram
from zeusdl.telebot.uploader import TelegramUploader
up = TelegramUploader(bot_token=BOT_TOKEN, channel=CHANNEL)
# up.upload_directory(OUTPUT_DIR)

# Se reconnecter BangBros (si cookies expires)
from zeusdl.bangbros_login import BangBrosLogin
# BangBrosLogin(COOKIES_PATH).login(BB_EMAIL, BB_PASSWORD)

print('Raccourcis charges. Decommente les lignes que tu veux executer.')